# Revisión de especies: confirmar o rechazar sugerencias

Interfaz rústica dentro del propio notebook (`ipywidgets`, no requiere servidor ni app aparte) para confirmar o rechazar, una por una, las sugerencias de especie que ya dejó `Especies_PlantNet.ipynb` en estado `pendiente`.

**No es entrenamiento de modelo.** Es revisión humana en el bucle: el modelo propone, tú decides, se registra la decisión. Lo rechazado no se pierde, vuelve a la cola de "sin identificar" para la siguiente pasada (otro modelo, iNaturalist, o a mano si al final quedan pocas).

**Insumo:** `../../experimentos/bd/campuslab_volumen.db` (la misma base de siempre, sin mover ni duplicar).

In [ ]:
import json
import sqlite3
import pandas as pd
from pathlib import Path

from IPython.display import display
import ipywidgets as widgets
import requests

def cargar_config(archivo="config_local.json"):
    ruta = Path(archivo)
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontro {archivo}. Copia config_local.example.json y ajustalo.")
    return json.loads(ruta.read_text(encoding="utf-8"))

config = cargar_config()
RUTA_BD = Path("../../experimentos/bd/campuslab_volumen.db")
con = sqlite3.connect(RUTA_BD)
print(f"Base de datos: {RUTA_BD.resolve()} ({'existe' if RUTA_BD.exists() else 'NO ENCONTRADA'})")

## 1. Conteo general: cuántas fotos tienen especie identificada

In [ ]:
total = con.execute("SELECT COUNT(*) FROM fotos").fetchone()[0]
con_especie = con.execute("SELECT COUNT(*) FROM fotos WHERE especie_id IS NOT NULL").fetchone()[0]
sin_especie = total - con_especie
pendientes_revision = con.execute("SELECT COUNT(*) FROM fotos WHERE estado = 'pendiente' AND especie_id IS NOT NULL").fetchone()[0]

print(f"Total de fotos en el catálogo: {total:,}")
print(f"Con especie identificada:      {con_especie:,} ({con_especie/total*100:.1f}%)")
print(f"Sin especie identificada:      {sin_especie:,} ({sin_especie/total*100:.1f}%)")
print(f"  de las cuales, esperando tu confirmación ahora mismo: {pendientes_revision:,}")

## 2. Esquema: columnas de taxonomía y tabla de auditoría de decisiones

In [ ]:
cols_especies = [r[1] for r in con.execute("PRAGMA table_info(especies)")]
for col in ("familia", "genero"):
    if col not in cols_especies:
        con.execute(f"ALTER TABLE especies ADD COLUMN {col} TEXT")

con.execute("""
    CREATE TABLE IF NOT EXISTS identificaciones_revisadas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        foto_id INTEGER NOT NULL REFERENCES fotos(id),
        especie_sugerida TEXT,
        score REAL,
        decision TEXT NOT NULL,  -- 'confirmada' / 'rechazada'
        revisor TEXT,
        fecha TEXT DEFAULT CURRENT_TIMESTAMP
    )
""")
con.commit()
print("Esquema listo: familia, genero, identificaciones_revisadas")

## 3. Cola de revisión

Una tarjeta a la vez: tu foto a la izquierda, la foto de referencia de la especie que sugirió Pl@ntNet a la derecha (viene incluida en su respuesta, no hace falta otra llamada), el score, y dos botones. Al decidir, pasa sola a la siguiente.

In [ ]:
REVISOR = config.get("nombre_revisor", "Remi")

def cargar_cola():
    # Sugerencias de PlantNet (ya con especie_id asignado, esperando confirmar)
    de_plantnet = con.execute(
        "SELECT f.id, f.ruta_actual, e.id AS especie_id, e.nombre_cientifico, e.plantnet_score, e.plantnet_raw "
        "FROM fotos f JOIN especies e ON e.id = f.especie_id "
        "WHERE f.estado = 'pendiente' ORDER BY f.id"
    ).fetchall()

    # Nombres reconstruidos o ambiguos de Resolucion_Nombres.ipynb (leidos directo del archivo
    # que ese notebook exporta, no viven en la BD hasta que se confirman aqui)
    ruta_no_resueltos = Path("../salida/Nombres_No_Resueltos.xlsx")
    de_resolucion = []
    if ruta_no_resueltos.exists():
        df_pend = pd.read_excel(ruta_no_resueltos)
        # Solo "posible_cientifico" por ahora: es un si/no claro (confirmar el nombre reconstruido).
        # "ambiguo" necesita elegir entre varias especies candidatas, es otro tipo de interaccion,
        # queda fuera de esta cola hasta que se diseñe ese flujo aparte.
        posibles = df_pend[df_pend["categoria"] == "posible_cientifico"]
        for _, fila in posibles.iterrows():
            ruta_foto = con.execute("SELECT ruta_actual FROM fotos WHERE id=?", (int(fila["id"]),)).fetchone()
            nombre_reconstruido = fila.get("nombre_cientifico")
            if ruta_foto and isinstance(nombre_reconstruido, str) and nombre_reconstruido.strip():
                de_resolucion.append((int(fila["id"]), ruta_foto[0], None, nombre_reconstruido, None, "{}"))

    return list(de_plantnet) + de_resolucion

def extraer_url_referencia(plantnet_raw_json):
    """Saca la primera imagen de referencia que trae la respuesta de Pl@ntNet, si existe."""
    try:
        cuerpo = json.loads(plantnet_raw_json)
        img = cuerpo["results"][0]["images"][0]
        return img.get("url", {}).get("o") or img.get("url", {}).get("m")
    except (KeyError, IndexError, TypeError, json.JSONDecodeError):
        return None

def registrar_decision(foto_id, especie_nombre, score, decision):
    con.execute(
        "INSERT INTO identificaciones_revisadas (foto_id, especie_sugerida, score, decision, revisor) VALUES (?,?,?,?,?)",
        (foto_id, especie_nombre, score, decision, REVISOR)
    )

def confirmar(foto_id, especie_id, nombre_cientifico=None):
    # Si la foto no traia especie_id todavia (viene de Resolucion_Nombres, no de PlantNet),
    # se busca o se crea la especie por nombre antes de enlazarla.
    if especie_id is None and nombre_cientifico:
        fila = con.execute("SELECT id FROM especies WHERE nombre_cientifico=?", (nombre_cientifico,)).fetchone()
        if fila:
            especie_id = fila[0]
        else:
            cur = con.execute(
                "INSERT INTO especies (nombre_cientifico, sugerido_por) VALUES (?,?)",
                (nombre_cientifico, "Resolucion_Nombres (nombre reconstruido)")
            )
            especie_id = cur.lastrowid
    con.execute("UPDATE fotos SET estado='usable', especie_id=? WHERE id=?", (especie_id, foto_id))
    con.execute(
        "UPDATE especies SET fuente_validacion=? WHERE id=? AND fuente_validacion IS NULL",
        ("revisión manual Campus Lab", especie_id)
    )
    con.commit()
    return especie_id

def rechazar(foto_id):
    # Vuelve a la cola de "sin identificar" para una futura pasada (otro modelo, iNaturalist, o a mano).
    # No se reintenta con la misma sugerencia: eso ya quedó registrado en identificaciones_revisadas.
    con.execute("UPDATE fotos SET especie_id=NULL, estado='pendiente_revision' WHERE id=?", (foto_id,))
    con.commit()

print(f"Revisor configurado: {REVISOR}")
print(f"En cola ahora mismo: {len(cargar_cola())}")

In [ ]:
# --- Revisión con input() de texto: mas confiable que botones interactivos entre distintos entornos ---
from IPython.display import display, Image as IPImage, clear_output

cola = cargar_cola()
print(f"En cola: {len(cola)}\n")

for foto_id, ruta_actual, especie_id, nombre, score, raw in cola:
    clear_output(wait=True)
    print(f"Sugerencia: {nombre}" + (f"  (score {score:.2f})" if score is not None else ""))
    print(f"Foto: {ruta_actual}\n")

    try:
        display(IPImage(filename=ruta_actual, width=350))
    except Exception as e:
        print(f"(no se pudo mostrar tu foto: {e})")

    url_ref = extraer_url_referencia(raw)
    if url_ref:
        try:
            display(IPImage(data=requests.get(url_ref, timeout=15).content, width=350))
        except Exception:
            print("(no se pudo cargar la foto de referencia)")

    respuesta = input("\n[C]onfirmar / [R]echazar / [P]osponer / [S]alir: ").strip().lower()

    if respuesta.startswith("c"):
        confirmar(foto_id, especie_id, nombre_cientifico=nombre)
        registrar_decision(foto_id, nombre, score, "confirmada")
    elif respuesta.startswith("r"):
        rechazar(foto_id)
        registrar_decision(foto_id, nombre, score, "rechazada")
    elif respuesta.startswith("s"):
        print("Sesión pausada, tu progreso ya quedó guardado.")
        break
    else:
        # Posponer: no se toca nada, simplemente pasa a la siguiente sin registrar decisión.
        # La misma sugerencia volverá a aparecer la próxima vez que corras esta celda.
        continue

print("\nListo por ahora. Vuelve a correr esta celda para seguir donde te quedaste (lo pospuesto reaparece, lo decidido no).")